# Kaggle runner — E1 separated feature selection + classification (precomputed 5×5)

Notebook dùng **hai input Kaggle Dataset**: `experiment-data-perm` chứa expression/metadata và `e1_repeated_splits_5x5` chứa ba `repeated_splits.csv`. CSV chỉ định 25 outer train/test split (5 repeat × 5 fold). Feature selection chỉ thấy train; phân loại tune inner-CV trong train và chấm outer-test đúng một lần.

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys
import pandas as pd

INPUT_DATASET = Path('/kaggle/input/experiment-data-perm')
INPUT_SPLITS_DATASET = Path('/kaggle/input/e1-repeated-splits-5x5')  # sửa nếu Kaggle mount slug khác
REPO_URL = 'https://github.com/datachain-uit/RNA-Seq-FS-Survey.git'
REPO_BRANCH = 'survey-v2-code'
WORK_ROOT = Path('/kaggle/working/kltn_data')
COHORTS = ['gse68465_stage_early_late', 'tcga_luad_stage_early_late', 'tcga_lusc_stage_early_late']
K_GRID = [10, 50, 100, 200, 500]
RUN_FILTER_EMBEDDED = True   # False nếu chỉ muốn chạy GA/BPSO
RUN_FULL_CLASSIFIERS = True  # classifier cho panel filter/embedded
RUN_WRAPPERS = False         # GA/BPSO là job tách riêng
RUN_WRAPPER_CLASSIFIERS = True  # classifier cho panel GA/BPSO
# RUN_WRAPPERS: một Save Version = một cohort × một wrapper.
TARGET_COHORT = 'gse68465_stage_early_late'
TARGET_WRAPPER_METHOD = 'ga'  # 'ga' hoặc 'bpso'
TARGET_REPEATS = [0, 1, 2, 3, 4]  # có thể dùng [0, 1] cho job ngắn hơn
# Nếu tiếp tục một job dang dở: attach ZIP output của Version trước và điền đường dẫn này.
PREVIOUS_PROGRESS_ARCHIVE = None  # ví dụ Path('/kaggle/input/e1-ga-gse-output/e1_gse68465_stage_early_late_ga_output.zip')


In [ ]:
# Gate 1: xác nhận hai input và 25 split cố định/cohort trước khi chạy.
assert INPUT_DATASET.is_dir(), f'Không thấy expression input: {INPUT_DATASET}'
assert INPUT_SPLITS_DATASET.is_dir(), f'Không thấy split input: {INPUT_SPLITS_DATASET}'
for cohort in COHORTS:
    root = INPUT_DATASET / cohort
    for filename in ('expression.h5', 'metadata.csv'):
        assert (root / filename).is_file(), f'Thiếu {cohort}/{filename}'
    split_path = INPUT_SPLITS_DATASET / cohort / 'repeated_splits.csv'
    assert split_path.is_file(), f'Thiếu split: {split_path}'
    split = pd.read_csv(split_path)
    assert set(split.role) == {'train', 'test'}
    assert split[['repeat', 'fold']].drop_duplicates().shape[0] == 25
    assert (split[split.role == 'test'].groupby(['repeat', 'sample_id']).size() == 1).all()
    assert not (split[split.role == 'train'].merge(split[split.role == 'test'], on=['repeat', 'fold', 'patient_id'])).shape[0]
print('INPUT_AND_SPLITS_OK')


In [ ]:
# Gate 2: clone đúng commit đã push, cài dependencies, rồi liên kết input read-only (không copy H5).
REPO_DIR = Path('/kaggle/working/RNA-Seq-FS-Survey')
if REPO_DIR.exists(): shutil.rmtree(REPO_DIR)
subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, '--single-branch', REPO_URL, str(REPO_DIR)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], cwd=REPO_DIR, check=True)
commit_hash = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip()
WORK_ROOT.mkdir(parents=True, exist_ok=True)
if PREVIOUS_PROGRESS_ARCHIVE is not None:
    assert PREVIOUS_PROGRESS_ARCHIVE.is_file(), f'Không thấy archive resume: {PREVIOUS_PROGRESS_ARCHIVE}'
    shutil.unpack_archive(str(PREVIOUS_PROGRESS_ARCHIVE), str(WORK_ROOT))
    print('RESTORED_PROGRESS', PREVIOUS_PROGRESS_ARCHIVE)
link = WORK_ROOT / 'experiment-data-perm'
if not link.exists(): link.symlink_to(INPUT_DATASET, target_is_directory=True)
assert link.resolve() == INPUT_DATASET.resolve()
os.environ['KLTN_DATA_ROOT'] = str(WORK_ROOT)
print('COMMIT', commit_hash)


In [ ]:
# Phase A — filter + embedded. Không có classifier ở cell này.
if RUN_FILTER_EMBEDDED:
    subprocess.run([sys.executable, 'scripts/run_e1_feature_selection.py', '--datasets', *COHORTS, '--k', *map(str, K_GRID), '--splits-root', str(INPUT_SPLITS_DATASET)], cwd=REPO_DIR, env=os.environ.copy(), check=True)


In [ ]:
# Phase B — đọc fixed panels của Phase A; inner-CV chỉ dùng outer train.
if RUN_FILTER_EMBEDDED and RUN_FULL_CLASSIFIERS:
    subprocess.run([sys.executable, 'scripts/run_e1_classification.py', '--datasets', *COHORTS, '--splits-root', str(INPUT_SPLITS_DATASET)], cwd=REPO_DIR, env=os.environ.copy(), check=True)


In [ ]:
# Job RIÊNG: mỗi Save Version chỉ chạy một cohort × một wrapper; checkpoint sau mỗi fold.
if RUN_WRAPPERS:
    subprocess.run([sys.executable, 'scripts/run_e1_feature_selection.py', '--datasets', TARGET_COHORT, '--methods', TARGET_WRAPPER_METHOD, '--repeats', *map(str, TARGET_REPEATS), '--k', *map(str, K_GRID), '--splits-root', str(INPUT_SPLITS_DATASET), '--out', 'E1_wrapper_feature_selection'], cwd=REPO_DIR, env=os.environ.copy(), check=True)
    if RUN_WRAPPER_CLASSIFIERS:
        subprocess.run([sys.executable, 'scripts/run_e1_classification.py', '--datasets', TARGET_COHORT, '--methods', TARGET_WRAPPER_METHOD, '--splits-root', str(INPUT_SPLITS_DATASET), '--selection-root', str(WORK_ROOT / 'results/survey_v2/E1_wrapper_feature_selection'), '--out', 'E1_wrapper_classification'], cwd=REPO_DIR, env=os.environ.copy(), check=True)


In [ ]:
# Gate 3: chỉ save/download output nếu provenance đầy đủ.
result_root = WORK_ROOT / 'results/survey_v2'
configs = list(result_root.glob('E1_*/*/config.json'))
assert configs, 'Không tìm thấy config output'
for path in configs:
    cfg = json.loads(path.read_text(encoding='utf-8'))
    assert cfg.get('traceable') is True and cfg.get('commit_hash') == commit_hash, path
archive = Path('/kaggle/working') / f'e1_{TARGET_COHORT}_{TARGET_WRAPPER_METHOD}_output.zip'
shutil.make_archive(str(archive.with_suffix('')), 'zip', root_dir=WORK_ROOT, base_dir='results')
assert archive.is_file() and archive.stat().st_size > 0
print('E1_OK', result_root, 'ARCHIVE=', archive)
